In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
import re
from pathlib import Path

iemocap_path = Path("/kaggle/input/datasets/dejolilandry/iemocapfullrelease/IEMOCAP_full_release")

emotions = {}
for emo_file in iemocap_path.rglob("dialog/EmoEvaluation/*.txt"):
    with open(emo_file) as f:
        for line in f:
            if line.startswith("["):
                match = re.match(r'\[.*?\]\s+\S+\s+(\w+)', line)
                if match:
                    emo = match.group(1)
                    emotions[emo] = emotions.get(emo, 0) + 1

print("=== IEMOCAP Emotions ===\n")
for emo, count in sorted(emotions.items(), key=lambda x: -x[1]):
    bar = "█" * (count // 50)
    print(f"  {emo:<10} {count:>5}  {bar}")
print(f"\n  {'TOTAL':<10} {sum(emotions.values()):>5}")

In [ ]:
from pathlib import Path
import os

iemocap_path = Path("/kaggle/input/datasets/dejolilandry/iemocapfullrelease/IEMOCAP_full_release")

wav_sentences = list(iemocap_path.rglob("sentences/wav/*/*.wav"))
wav_dialog = list(iemocap_path.rglob("dialog/wav/*.wav"))
avi_files = list(iemocap_path.rglob("*.avi"))
mocap_head = list(iemocap_path.rglob("MOCAP_head/*.txt"))
mocap_hand = list(iemocap_path.rglob("MOCAP_hand/*.txt"))
transcripts = list(iemocap_path.rglob("transcriptions/*.txt"))

print("=== IEMOCAP Modalities ===\n")
print(f"  Audio (sentences/wav):  {len(wav_sentences):>6} files")
print(f"  Audio (dialog/wav):     {len(wav_dialog):>6} files")
print(f"  Video (avi):            {len(avi_files):>6} files")
print(f"  MOCAP head:             {len(mocap_head):>6} files")
print(f"  MOCAP hand:             {len(mocap_hand):>6} files")
print(f"  Transcriptions:         {len(transcripts):>6} files")

print("\n=== Sample WAV filename ===")
if wav_sentences:
    sample = wav_sentences[0]
    print(f"  {sample.name}")
    print(f"  Utterance ID: {sample.stem}")

In [ ]:
from pathlib import Path
import re

iemocap_path = Path("/kaggle/input/datasets/rumaiya/iemocap/IEMOCAP_full_release/IEMOCAP_full_release")

emo_file = iemocap_path / "Session1/dialog/EmoEvaluation/Ses01F_impro01.txt"
print("=== Emotion Label File ===")
print(f"File: {emo_file.name}\n")
with open(emo_file) as f:
    for i, line in enumerate(f):
        if line.startswith("["):
            print(line.strip())
        if i >= 8:
            break

dialog_avi = iemocap_path / "Session1/dialog/avi/DivX/Ses01F_impro01.avi"
print(f"\n=== Dialog Video ===")
print(f"Exists: {dialog_avi.exists()}")
if dialog_avi.exists():
    print(f"File: {dialog_avi.name}")
    print(f"Size: {dialog_avi.stat().st_size / (1024*1024):.1f} MB")

utt_wav = iemocap_path / "Session1/sentences/wav/Ses01F_impro01/Ses01F_impro01_F000.wav"
print(f"\n=== Utterance Audio ===")
print(f"Exists: {utt_wav.exists()}")
if utt_wav.exists():
    print(f"File: {utt_wav.name}")
    print(f"Size: {utt_wav.stat().st_size / 1024:.1f} KB")

trans_file = iemocap_path / "Session1/dialog/transcriptions/Ses01F_impro01.txt"
print(f"\n=== Transcription ===")
print(f"Exists: {trans_file.exists()}")
if trans_file.exists():
    with open(trans_file) as f:
        for i, line in enumerate(f):
            print(line.strip())
            if i >= 5:
                break

sessions = list(iemocap_path.glob("Session*"))
print(f"\n=== Sessions ===")
print(f"Total: {len(sessions)}")
for s in sorted(sessions):
    avis = list((s / "dialog/avi/DivX").glob("*.avi"))
    wavs = list((s / "sentences/wav").rglob("*.wav"))
    print(f"  {s.name}: {len(avis)} videos, {len(wavs)} audio files")

# Preprocessing

In [ ]:
import re
from pathlib import Path
from collections import Counter

iemocap_path = Path("/kaggle/input/datasets/rumaiya/iemocap/IEMOCAP_full_release")

# Count all emotions with encoding fix
emotions = Counter()
for emo_file in iemocap_path.rglob("dialog/EmoEvaluation/*.txt"):
    try:
        with open(emo_file, encoding='utf-8', errors='ignore') as f:
            for line in f:
                if line.startswith("["):
                    match = re.match(r'\[.*?\]\s+\S+\s+(\w+)', line)
                    if match:
                        emotions[match.group(1)] += 1
    except:
        continue

print("=== All Emotions ===\n")
for emo, count in sorted(emotions.items(), key=lambda x: -x[1]):
    print(f"  {emo:<10} {count:>5}")

USABLE_EMOTIONS = {'neu', 'ang', 'sad', 'hap', 'exc', 'fru'}
EMOTION_TO_IDX = {e: i for i, e in enumerate(sorted(USABLE_EMOTIONS))}

usable_count = sum(emotions[e] for e in USABLE_EMOTIONS)
print(f"\n=== Usable (6 emotions): {usable_count} ===")
print(f"Emotion mapping: {EMOTION_TO_IDX}")
print(" Cell 0 done!")

In [ ]:
import pandas as pd
import re
from pathlib import Path

iemocap_path = Path("/kaggle/input/datasets/rumaiya/iemocap/IEMOCAP_full_release/IEMOCAP_full_release")

USABLE_EMOTIONS = {'neu', 'ang', 'sad', 'hap', 'exc', 'fru'}
EMOTION_TO_IDX = {e: i for i, e in enumerate(sorted(USABLE_EMOTIONS))}

data = []

for session in sorted(iemocap_path.glob("Session*")):
    emo_dir = session / "dialog" / "EmoEvaluation"
    trans_dir = session / "dialog" / "transcriptions"
    wav_dir = session / "sentences" / "wav"
    video_dir = session / "dialog" / "avi" / "DivX"
    
    for emo_file in sorted(emo_dir.glob("*.txt")):
        dialog_id = emo_file.stem
        trans_file = trans_dir / f"{dialog_id}.txt"
        video_file = video_dir / f"{dialog_id}.avi"
        
        transcriptions = {}
        if trans_file.exists():
            try:
                with open(trans_file, encoding='utf-8', errors='ignore') as f:
                    for line in f:
                        match = re.match(r'(\S+)\s+\[.*?\]:\s+(.+)', line)
                        if match:
                            transcriptions[match.group(1)] = match.group(2).strip()
            except:
                pass
        
        try:
            with open(emo_file, encoding='utf-8', errors='ignore') as f:
                for line in f:
                    if line.startswith("["):
                        match = re.match(r'\[([\d.]+)\s+-\s+([\d.]+)\]\s+(\S+)\s+(\w+)', line)
                        if match:
                            start, end, utt_id, emotion = match.groups()
                            
                            if emotion not in USABLE_EMOTIONS:
                                continue
                            
                            wav_file = wav_dir / dialog_id / f"{utt_id}.wav"
                            if not wav_file.exists():
                                continue
                            
                            data.append({
                                'utterance_id': utt_id,
                                'dialog_id': dialog_id,
                                'session': session.name,
                                'wav_path': str(wav_file),
                                'video_path': str(video_file) if video_file.exists() else None,
                                'start_time': float(start),
                                'end_time': float(end),
                                'emotion': emotion,
                                'text': transcriptions.get(utt_id, "")
                            })
        except:
            continue

df_iemocap = pd.DataFrame(data)
print(f"Total samples: {len(df_iemocap)}")
if len(df_iemocap) > 0:
    print(f"With video: {df_iemocap['video_path'].notna().sum()}")
    print(f"\nEmotion distribution:")
    print(df_iemocap['emotion'].value_counts())
print(" Cell 1 done!")

In [ ]:
import cv2
import numpy as np
import librosa
import soundfile as sf
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

iemocap_output = Path("/kaggle/working/iemocap_features")
iemocap_output.mkdir(exist_ok=True)
(audio_dir := iemocap_output / "audio").mkdir(exist_ok=True)
(visual_dir := iemocap_output / "visual").mkdir(exist_ok=True)
(pose_dir := iemocap_output / "pose").mkdir(exist_ok=True)
(text_dir := iemocap_output / "text").mkdir(exist_ok=True)

face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')

def detect_face(frame):
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    faces = face_cascade.detectMultiScale(gray, 1.3, 3, minSize=(20, 20))
    if len(faces) > 0:
        return max(faces, key=lambda f: f[2] * f[3])
    return None

print(f"Extracting features from {len(df_iemocap)} samples...")
success = 0; fail = 0
face_detected = 0; face_missing = 0

for idx, row in tqdm(df_iemocap.iterrows(), total=len(df_iemocap), desc="IEMOCAP"):
    utt_id = row['utterance_id']
    
    try:
        y, sr = sf.read(row['wav_path'])
        if y.ndim > 1:
            y = y.mean(axis=1)
        if sr != 16000:
            y = librosa.resample(y, orig_sr=sr, target_sr=16000)
            sr = 16000
        
        if len(y) < 1600:
            fail += 1; continue
        
        mel = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=128, hop_length=512)
        mel_db = librosa.power_to_db(mel, ref=np.max)
 
        mel_db = (mel_db - mel_db.mean()) / (mel_db.std() + 1e-8)
        mel_db = mel_db.astype(np.float32) 

        pitch = librosa.yin(y, fmin=75, fmax=600, sr=sr).astype(np.float32)
        rms   = librosa.feature.rms(y=y, hop_length=512)[0].astype(np.float32)
        
        np.savez_compressed(audio_dir / f"{utt_id}.npz", mel=mel_db, pitch=pitch, rms=rms)
        
        np.save(text_dir / f"{utt_id}.npy", np.array([row['text']], dtype=object))
        
        if pd.notna(row['video_path']):
            cap = cv2.VideoCapture(row['video_path'])
            fps = cap.get(cv2.CAP_PROP_FPS)
            
            start_frame = int(row['start_time'] * fps)
            end_frame   = int(row['end_time']   * fps)
            total_frames = end_frame - start_frame
            
            if total_frames <= 0:
                cap.release()
                fail += 1; continue
            
            step = max(total_frames // 8, 1)
            face_frames = []; pose_proxy = []
            
            for frame_num in range(start_frame, end_frame, step):
                cap.set(cv2.CAP_PROP_POS_FRAMES, frame_num)
                ret, frame = cap.read()
                if not ret: continue
                
                face_box = detect_face(frame)
                fh, fw   = frame.shape[:2]
                
                if face_box is not None:
                    x, yb, w, h = face_box
                    face_detected += 1  
                    pose_proxy.append([
                        (x + w/2) / fw,   # center x
                        (yb + h/2) / fh,  # center y
                        h / fh,           # relative height
                        w / max(h, 1)     # aspect ratio
                    ])
                    pad  = int(0.15 * h)
                    crop = frame[max(0, yb-pad):yb+h+pad, max(0, x-pad):x+w+pad]
                else:
                    face_missing += 1  
                    s    = min(fh, fw)
                    crop = frame[(fh-s)//2:(fh+s)//2, (fw-s)//2:(fw+s)//2]
                    pose_proxy.append([0.5, 0.5, 0.5, 1.0])
                
                if crop.size > 0:
                    crop = cv2.resize(crop, (112, 112))
                    face_frames.append(crop)
                
                if len(face_frames) >= 8: break
            
            cap.release()
            
            if face_frames:
                np.save(visual_dir / f"{utt_id}.npy",
                        np.array(face_frames, dtype=np.uint8))
                np.save(pose_dir   / f"{utt_id}.npy",
                        np.array(pose_proxy, dtype=np.float32))
        
        success += 1
    
    except Exception as e:
        fail += 1
        if fail <= 5:
            print(f"Error {utt_id}: {e}")

total_face_attempts = face_detected + face_missing
face_rate = 100 * face_detected / total_face_attempts if total_face_attempts > 0 else 0

print(f"\n Done! Success: {success} | Failed: {fail}")
print(f"Face detection rate: {face_detected}/{total_face_attempts} ({face_rate:.1f}%)")

if face_rate < 70:
    print("  WARNING: Low face detection rate — visual features may be noisy!")
else:
    print(" Face detection rate is acceptable")

df_iemocap.to_csv(iemocap_output / "iemocap_metadata.csv", index=False)

print(f"\nAudio:  {len(list(audio_dir.glob('*.npz')))}")
print(f"Visual: {len(list(visual_dir.glob('*.npy')))}")
print(f"Pose:   {len(list(pose_dir.glob('*.npy')))}")
print(f"Text:   {len(list(text_dir.glob('*.npy')))}")

# Zip

In [ ]:
!zip -r /kaggle/working/iemocap_features.zip /kaggle/working/iemocap_features/

print(" Features zipped!")
print("\n Download: iemocap_features.zip from Output tab")
print(" Size:", f"{Path('/kaggle/working/iemocap_features.zip').stat().st_size / (1024**2):.1f} MB")
print("\n To use next time:")
print("   !unzip -q /kaggle/input/your-dataset/iemocap_features.zip -d /kaggle/working/")

# Load

In [ ]:
from pathlib import Path

iemocap_output = Path("/kaggle/input/datasets/ramspimatiz7gmailcom/iemocap-features/kaggle/working/iemocap_features")

print(f"Audio:  {len(list((iemocap_output/'audio').glob('*.npz')))}")
print(f"Visual: {len(list((iemocap_output/'visual').glob('*.npy')))}")
print(f"Pose:   {len(list((iemocap_output/'pose').glob('*.npy')))}")
print(f"Text:   {len(list((iemocap_output/'text').glob('*.npy')))}")
print(" Ready to use!")

# Model Definition & Training Phase

In [ ]:
import os, copy, time, warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, classification_report, confusion_matrix
from pathlib import Path
from transformers import AutoTokenizer, AutoModel
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights
import matplotlib.pyplot as plt
import seaborn as sns
warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
torch.manual_seed(42)
np.random.seed(42)

iemocap_output = Path("/kaggle/input/datasets/satishkagglist/iemocap-features-new/kaggle/working/iemocap_features")
audio_dir = iemocap_output / "audio"
visual_dir = iemocap_output / "visual"
pose_dir = iemocap_output / "pose"
text_dir = iemocap_output / "text"

metadata = pd.read_csv(iemocap_output / "iemocap_metadata.csv")
print(f"Total: {len(metadata)}")

emotion_to_idx = {'ang': 0, 'exc': 1, 'fru': 2, 'hap': 3, 'neu': 4, 'sad': 5}
idx_to_emotion = {v: k for k, v in emotion_to_idx.items()}

valid = []
for _, row in metadata.iterrows():
    uid = row['utterance_id']
    if ((audio_dir / f"{uid}.npz").exists() and
        (visual_dir / f"{uid}.npy").exists() and
        (pose_dir / f"{uid}.npy").exists()):
        valid.append(row)

metadata = pd.DataFrame(valid).reset_index(drop=True)
print(f"Valid: {len(metadata)}")

train_df, val_df = train_test_split(
    metadata, test_size=0.2, stratify=metadata['emotion'], random_state=42
)

print(f"\nTrain: {len(train_df)} | Val: {len(val_df)}")
print("\n=== Distribution ===")
for name, d in [("Train", train_df), ("Val", val_df)]:
    print(f"\n{name}:")
    for emo, count in d['emotion'].value_counts().items():
        print(f"  {emo:<4} {count:>5} ({100*count/len(d):.1f}%)")

tokenizer = AutoTokenizer.from_pretrained("microsoft/deberta-v3-base")
print("\n Cell 1 done!")

In [ ]:
IMAGENET_MEAN = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
IMAGENET_STD  = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)
MAX_FRAMES = 8
MAX_TEXT_LEN = 64
BATCH_SIZE = 16

def spec_augment(mel, F=15, T=40):
    mel = mel.clone()
    _, n_mels, time = mel.shape
    f0 = np.random.randint(0, max(n_mels - F, 1))
    mel[:, f0:f0+F, :] = 0
    if time > T:
        t0 = np.random.randint(0, time - T)
        mel[:, :, t0:t0+T] = 0
    return mel

def mixup_batch(batch, alpha=0.3):
    lam = np.random.beta(alpha, alpha)
    idx = torch.randperm(batch['label'].size(0))
    mixed = {}
    for k in ['mel', 'visual', 'pose']:
        mixed[k] = lam * batch[k] + (1-lam) * batch[k][idx]
    mixed['input_ids']      = batch['input_ids']
    mixed['attention_mask'] = batch['attention_mask']
    return mixed, batch['label'], batch['label'][idx], lam
    

class IEMOCAPDataset(Dataset):
    def __init__(self, df, train=True):
        self.df = df.reset_index(drop=True)
        self.train = train

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        uid = row['utterance_id']

        # Audio
        audio = np.load(audio_dir / f"{uid}.npz")
        mel = torch.from_numpy(audio['mel'].astype(np.float32))

        # Visual
        visual = np.load(visual_dir / f"{uid}.npy")
        T = len(visual)
        if T > MAX_FRAMES:
            indices = np.linspace(0, T-1, MAX_FRAMES, dtype=int)
            visual = visual[indices]
        elif T < MAX_FRAMES:
            visual = np.concatenate([visual, np.repeat(visual[-1:], MAX_FRAMES-T, axis=0)])

        visual = torch.from_numpy(visual).float() / 255.0
        visual = visual.permute(0, 3, 1, 2)
        visual = F.interpolate(visual, size=(224, 224), mode='bilinear', align_corners=False)
        visual = (visual - IMAGENET_MEAN) / IMAGENET_STD

        if self.train and torch.rand(1).item() > 0.5:
            visual = torch.flip(visual, dims=[3])

        # Pose
        pose = np.load(pose_dir / f"{uid}.npy")
        if len(pose) > MAX_FRAMES:
            pose = pose[np.linspace(0, len(pose)-1, MAX_FRAMES, dtype=int)]
        elif len(pose) < MAX_FRAMES:
            pose = np.concatenate([pose, np.repeat(pose[-1:], MAX_FRAMES-len(pose), axis=0)])
        pose = torch.from_numpy(pose.astype(np.float32))

        # Text
        text = str(row['text']) if pd.notna(row['text']) else ""
        tokens = tokenizer(text, max_length=MAX_TEXT_LEN, padding='max_length',
                           truncation=True, return_tensors='pt')
        input_ids = tokens['input_ids'].squeeze(0)
        attention_mask = tokens['attention_mask'].squeeze(0)

        label = torch.tensor(emotion_to_idx[row['emotion']], dtype=torch.long)

        return {'mel': mel, 'visual': visual, 'pose': pose,
                'input_ids': input_ids, 'attention_mask': attention_mask, 'label': label}


def collate_fn(batch):
    labels = torch.stack([b['label'] for b in batch])
    input_ids = torch.stack([b['input_ids'] for b in batch])
    attention_mask = torch.stack([b['attention_mask'] for b in batch])
    max_audio = max(b['mel'].shape[1] for b in batch)

    mels, visuals, poses = [], [], []
    for b in batch:
        mels.append(F.pad(b['mel'], (0, max_audio - b['mel'].shape[1])))
        visuals.append(b['visual'])
        poses.append(b['pose'])

    return {'mel': torch.stack(mels), 'visual': torch.stack(visuals),
            'pose': torch.stack(poses), 'input_ids': input_ids,
            'attention_mask': attention_mask, 'label': labels}


from torch.utils.data import WeightedRandomSampler

def make_weighted_sampler(df, emotion_to_idx):
    labels = df['emotion'].map(emotion_to_idx).values
    class_counts = np.bincount(labels)
    
    # Weight per class = 1 / count
    class_weights = 1.0 / class_counts
    sample_weights = class_weights[labels]
    
    return WeightedRandomSampler(
        weights=torch.tensor(sample_weights, dtype=torch.float32),
        num_samples=len(sample_weights),
        replacement=True  # allows oversampling minorities
    )

sampler = make_weighted_sampler(train_df, emotion_to_idx)

train_loader = DataLoader(
    IEMOCAPDataset(train_df, True),
    batch_size=BATCH_SIZE,
    sampler=sampler,        #
    num_workers=0,
    collate_fn=collate_fn,
    pin_memory=True
)

val_loader = DataLoader(IEMOCAPDataset(val_df, False), batch_size=BATCH_SIZE,
                         shuffle=False, num_workers=0, collate_fn=collate_fn, pin_memory=True)

batch = next(iter(train_loader))
print(f"mel:    {batch['mel'].shape}")
print(f"visual: {batch['visual'].shape}")
print(f"pose:   {batch['pose'].shape}")
print(f"ids:    {batch['input_ids'].shape}")
print(f"label:  {batch['label'].shape}")
print(" Cell 2 done!")

In [ ]:
class AudioEncoder(nn.Module):
    def __init__(self, n_mels=128, d_model=256):
        super().__init__()
        self.cnn = nn.Sequential(
            nn.Conv1d(n_mels, 128, 5, padding=2), nn.BatchNorm1d(128), nn.GELU(),
            nn.MaxPool1d(2), nn.Dropout(0.1),
            nn.Conv1d(128, 256, 5, padding=2), nn.BatchNorm1d(256), nn.GELU(),
            nn.MaxPool1d(2), nn.Dropout(0.1),
            nn.Conv1d(256, d_model, 3, padding=1), nn.BatchNorm1d(d_model), nn.GELU(),
            nn.AdaptiveAvgPool1d(16)
        )
        self.transformer = nn.TransformerEncoder(
            nn.TransformerEncoderLayer(d_model, 4, 512, 0.3, batch_first=True, activation='gelu'), 2
        )

    def forward(self, mel):
        return self.transformer(self.cnn(mel).permute(0, 2, 1))


class VisualEncoder(nn.Module):
    def __init__(self, d_model=256):
        super().__init__()
        eff = efficientnet_b0(weights=EfficientNet_B0_Weights.IMAGENET1K_V1)
        self.backbone = nn.Sequential(*list(eff.children())[:-2])
        self.pool = nn.AdaptiveAvgPool2d(1)
        for param in self.backbone.parameters():
            param.requires_grad = False
        for child in list(self.backbone[0].children())[-2:]:
            for p in child.parameters(): p.requires_grad = True

        self.proj = nn.Sequential(nn.Linear(1280, d_model), nn.LayerNorm(d_model), nn.GELU(), nn.Dropout(0.1))
        self.transformer = nn.TransformerEncoder(
            nn.TransformerEncoderLayer(d_model, 4, 512, 0.3, batch_first=True, activation='gelu'), 2
        )

    def forward(self, frames):
        B, T = frames.shape[:2]
        x = self.backbone(frames.reshape(B*T, *frames.shape[2:]))
        x = self.pool(x).squeeze(-1).squeeze(-1)
        return self.transformer(self.proj(x).reshape(B, T, -1))


class TextEncoder(nn.Module):
    def __init__(self, d_model=256):
        super().__init__()
        self.deberta = AutoModel.from_pretrained("microsoft/deberta-v3-base")
        for param in self.deberta.parameters():
            param.requires_grad = False
        self.proj = nn.Sequential(
            nn.LayerNorm(768), nn.Linear(768, d_model),
            nn.LayerNorm(d_model), nn.GELU(), nn.Dropout(0.1)
        )

    @torch.autocast(device_type='cuda', enabled=False)
    def forward(self, input_ids, attention_mask):
        out = self.deberta(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state
        return self.proj(out.float())



class PoseGating(nn.Module):
    def __init__(self, d_model=256):
        super().__init__()
        self.gate = nn.Sequential(nn.Linear(4, 128), nn.GELU(), nn.Dropout(0.2),
                                   nn.Linear(128, d_model), nn.Sigmoid())
        self.norm = nn.LayerNorm(d_model)

    def forward(self, visual, pose):
        return self.norm(visual * self.gate(pose))


class CrossModalLayer(nn.Module):
    def __init__(self, d_model=256):
        super().__init__()
        self.attn = nn.MultiheadAttention(d_model, 4, batch_first=True, dropout=0.2)
        self.norm1 = nn.LayerNorm(d_model)
        self.ffn = nn.Sequential(nn.Linear(d_model, d_model*2), nn.GELU(),
                                  nn.Dropout(0.2), nn.Linear(d_model*2, d_model))
        self.norm2 = nn.LayerNorm(d_model)

    def forward(self, query, context):
        x = self.norm1(query + self.attn(query, context, context)[0])
        return self.norm2(x + self.ffn(x))


class Disentangler(nn.Module):
    def __init__(self, d_model=256):
        super().__init__()
        self.prosody = nn.Sequential(nn.Linear(d_model, d_model), nn.GELU(), nn.Linear(d_model, d_model))
        self.semantic = nn.Sequential(nn.Linear(d_model, d_model), nn.GELU(), nn.Linear(d_model, d_model))

    def forward(self, audio, text):
        return self.prosody(audio), self.semantic(text)

    def ortho_loss(self, p, s):
        p = F.normalize(p.mean(1), dim=1)
        s = F.normalize(s.mean(1), dim=1)
        return torch.clamp((p * s).sum(1).abs().mean(), max=10.0)


print(" Cell 3 done!")

In [ ]:
class IEMOCAPModel(nn.Module):
    def __init__(self, n_mels=128, d_model=256, num_classes=6):
        super().__init__()
        self.audio_enc = AudioEncoder(n_mels, d_model)
        self.visual_enc = VisualEncoder(d_model)
        self.text_enc = TextEncoder(d_model)
        self.pose_gate = PoseGating(d_model)
        self.disentangler = Disentangler(d_model)

        self.av_fusion = CrossModalLayer(d_model)
        self.at_fusion = CrossModalLayer(d_model)
        self.vt_fusion = CrossModalLayer(d_model)

        self.fusion_norm = nn.LayerNorm(d_model * 3)
        self.fusion_proj = nn.Sequential(nn.Linear(d_model*3, d_model), nn.GELU(), nn.Dropout(0.4))
        self.classifier = nn.Sequential(nn.Linear(d_model, 128), nn.GELU(), nn.Dropout(0.3),
                                         nn.Linear(128, num_classes))
        self.embedding_dim = d_model

    def forward(self, mel, visual, pose, input_ids, attention_mask):
        audio_feat = self.audio_enc(mel)
        visual_feat = self.visual_enc(visual)
        text_feat = self.text_enc(input_ids, attention_mask)

        visual_feat = self.pose_gate(visual_feat, pose)
        prosody, semantic = self.disentangler(audio_feat, text_feat)

        fused_av = self.av_fusion(prosody, visual_feat)
        fused_at = self.at_fusion(prosody, semantic)
        fused_vt = self.vt_fusion(visual_feat, semantic)

        combined = torch.cat([fused_av.mean(1), fused_at.mean(1), fused_vt.mean(1)], dim=1)
        embedding = self.fusion_proj(self.fusion_norm(combined))
        logits = self.classifier(embedding)

        return logits, embedding, prosody, semantic


model = IEMOCAPModel(128, 256, 6).to(device)
print(f"Total: {sum(p.numel() for p in model.parameters()):,}")
print(f"Trainable: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

with torch.no_grad():
    logits, emb, p, s = model(
        batch['mel'].to(device), batch['visual'].to(device),
        batch['pose'].to(device), batch['input_ids'].to(device),
        batch['attention_mask'].to(device))
print(f"Logits: {logits.shape} nan={logits.isnan().any()}")
print(" Cell 4 done!")

In [ ]:
from sklearn.utils.class_weight import compute_class_weight

class_weights_np = compute_class_weight(
    'balanced',
    classes=np.arange(6),
    y=train_df['emotion'].map(emotion_to_idx).values
)
CLASS_WEIGHTS = torch.tensor(class_weights_np, dtype=torch.float32).to(device)
print("Class weights:", {idx_to_emotion[i]: f"{w:.3f}" for i, w in enumerate(class_weights_np)})


class FocalLoss(nn.Module):
    def __init__(self, gamma=2.0, weight=None):
        super().__init__()
        self.gamma = gamma
        self.weight = weight  # per-class weights, not scalar alpha

    def forward(self, inputs, targets):
        # Get weighted CE loss per sample
        ce_loss = F.cross_entropy(
            inputs, targets,
            weight=self.weight,   # per-class weights handle imbalance
            reduction='none'
        )
        p_t = torch.exp(-ce_loss)
        # gamma focuses on hard examples — no alpha scalar needed
        focal_loss = (1 - p_t) ** self.gamma * ce_loss
        return focal_loss.mean()

# Usage
focal_fn = FocalLoss(gamma=2.0, weight=CLASS_WEIGHTS).to(device)


class CenterLoss(nn.Module):
    def __init__(self, num_classes, feat_dim):
        super().__init__()
        self.centers = nn.Parameter(torch.randn(num_classes, feat_dim))

    def forward(self, features, labels):
        batch_size = features.size(0)
        centers_batch = self.centers[labels]
        return ((features - centers_batch) ** 2).sum() / (2.0 * batch_size)

print(" Cell 5 done!")

In [ ]:
for p in model.visual_enc.backbone.parameters():
    p.requires_grad = False

EPOCHS   = 150
PATIENCE = 20
CENTER_W = 0.01
ORTHO_W  = 0.05

focal_fn  = FocalLoss(gamma=2.0, weight=CLASS_WEIGHTS).to(device)
center_fn = CenterLoss(6, 256).to(device)

deberta_params = [p for n, p in model.named_parameters()
                  if 'deberta' in n and p.requires_grad]
other_params   = [p for n, p in model.named_parameters()
                  if 'deberta' not in n and p.requires_grad]

print(f"DeBERTa trainable: {sum(p.numel() for p in deberta_params):,}")
print(f"Other trainable  : {sum(p.numel() for p in other_params):,}")

optimizer = torch.optim.AdamW([
    {'params': other_params,        'lr': 2e-4, 'weight_decay': 0.05},
    {'params': deberta_params,      'lr': 5e-5, 'weight_decay': 0.01},
    {'params': [center_fn.centers], 'lr': 2e-4, 'weight_decay': 0.0},
])

scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
    optimizer, T_0=20, T_mult=1, eta_min=1e-6
)

print(f"Trainable: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")
print(" Cell 6 Ready!")

In [ ]:
best_f1=0; best_acc=0; best_epoch=0; best_state=None; pat=0
history = {'tr_loss':[], 'tr_acc':[], 'tr_f1':[], 'vl_loss':[], 'vl_acc':[], 'vl_f1':[]}

print("=" * 80)
print(f"{'Ep':>3} | {'TrLoss':>7} {'TrAcc':>7} {'TrF1':>6} | {'VLoss':>7} {'VAcc':>7} {'VF1':>6} | {'Time':>5}")
print("=" * 80)

start = time.time()
for epoch in range(1, EPOCHS+1):
    t0 = time.time()
    model.train()
    tl=0; tc=0; tt=0; tp=[]; ty=[]

    for b in train_loader:
        b_dev = {k: v.to(device) for k, v in b.items()}
        lab   = b_dev['label']

        use_mix = epoch < int(0.75 * EPOCHS) and np.random.random() > 0.4
        if use_mix:
            mixed, ya, yb, lam = mixup_batch(b_dev, 0.3)
            logits, emb, pros, sem = model(
                mixed['mel'], mixed['visual'], mixed['pose'],
                mixed['input_ids'].long(), mixed['attention_mask'].long())
            loss = lam * focal_fn(logits, ya) + (1-lam) * focal_fn(logits, yb)
            loss += CENTER_W * center_fn(emb, ya)
        else:
            b_dev['mel'] = spec_augment(b_dev['mel'])
            logits, emb, pros, sem = model(
                b_dev['mel'], b_dev['visual'], b_dev['pose'],
                b_dev['input_ids'], b_dev['attention_mask'])
            loss = (focal_fn(logits, lab)
                  + CENTER_W * center_fn(emb, lab)
                  + ORTHO_W  * model.disentangler.ortho_loss(pros.float(), sem.float()))

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step(epoch + len(train_loader))  #  CosineWarmRestarts needs this

        bs = lab.size(0); tl += loss.item() * bs
        preds = logits.argmax(1); tc += (preds == lab).sum().item()
        tt += bs; tp.extend(preds.cpu().numpy()); ty.extend(lab.cpu().numpy())

    tr_l=tl/tt; tr_a=tc/tt; tr_f=f1_score(ty, tp, average='weighted')

    model.eval()
    tl=0; tc=0; tt=0; tp=[]; ty=[]
    with torch.no_grad():
        for b in val_loader:
            logits, emb, _, _ = model(
                b['mel'].to(device), b['visual'].to(device),
                b['pose'].to(device), b['input_ids'].to(device),
                b['attention_mask'].to(device))
            lab  = b['label'].to(device)
            loss = focal_fn(logits, lab) + CENTER_W * center_fn(emb, lab)
            bs   = lab.size(0); tl += loss.item() * bs
            preds = logits.argmax(1); tc += (preds == lab).sum().item()
            tt += bs; tp.extend(preds.cpu().numpy()); ty.extend(lab.cpu().numpy())

    vl_l=tl/tt; vl_a=tc/tt; vl_f=f1_score(ty, tp, average='weighted')
    dt=time.time()-t0

    history['tr_loss'].append(tr_l); history['tr_acc'].append(tr_a); history['tr_f1'].append(tr_f)
    history['vl_loss'].append(vl_l); history['vl_acc'].append(vl_a); history['vl_f1'].append(vl_f)

    mk = ""
    if vl_f > best_f1:
        best_f1=vl_f; best_acc=vl_a; best_epoch=epoch
        best_state=copy.deepcopy(model.state_dict()); pat=0
        mk = " -"  #  Fixed marker
    else:
        pat += 1

    if epoch%3==0 or mk or epoch==1:
        print(f"{epoch:>3} | {tr_l:>7.4f} {tr_a:>7.4f} {tr_f:>6.4f} | "
              f"{vl_l:>7.4f} {vl_a:>7.4f} {vl_f:>6.4f} | {dt:>4.0f}s{mk} [{len(set(tp))}c]")

    if pat >= PATIENCE:
        print(f"\n Early stop at epoch {epoch}")
        break

print("=" * 80)
print(f" Done in {(time.time()-start)/60:.1f} min | Best: Ep{best_epoch} Acc={best_acc:.4f} F1={best_f1:.4f}")
model.load_state_dict(best_state)
torch.save(best_state, "/kaggle/working/best_iemocap_v3.pt")
print(" Saved!")

In [ ]:
# Final evaluation
model.eval()
all_preds, all_labels = [], []
with torch.no_grad():
    for b in val_loader:
        logits, _, _, _ = model(b['mel'].to(device), b['visual'].to(device),
                                 b['pose'].to(device), b['input_ids'].to(device),
                                 b['attention_mask'].to(device))
        preds = logits.argmax(1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(b['label'].numpy())

print("\n=== Classification Report ===")
print(classification_report(all_labels, all_preds, 
                           target_names=list(emotion_to_idx.keys()),
                           digits=4))

In [ ]:
# Confusion Matrix
cm = confusion_matrix(all_labels, all_preds)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=list(emotion_to_idx.keys()),
            yticklabels=list(emotion_to_idx.keys()))
plt.title('Confusion Matrix')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.savefig('/kaggle/working/confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

# Training History
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
metrics = [('loss', 'Loss'), ('acc', 'Accuracy'), ('f1', 'F1 Score')]
for ax, (metric, title) in zip(axes, metrics):
    ax.plot(history[f'tr_{metric}'], label='Train', linewidth=2)
    ax.plot(history[f'vl_{metric}'], label='Val', linewidth=2)
    ax.set_xlabel('Epoch')
    ax.set_ylabel(title)
    ax.set_title(f'{title} over Epochs')
    ax.legend()
    ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('/kaggle/working/training_history.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n All done!")